# M1 Macrophage Cell-Surface RNA (csRNA-seq) Biomarker Discovery

**Objective:** To identify robust, surface-localized RNA biomarkers (csRNAs) unique to the activated M1 macrophage state. 

## 1. Experimental Design & Biological Context
This pipeline analyzes sequencing data comparing two distinct macrophage states extracted via two fundamentally different physical methods:
* **Control (Resting):** Bone Marrow-Derived Macrophages (BMDM) - Extracted via **Trizol** (Whole-cell lysis; captures all internal/external RNA).
* **Experimental (Activated):** M1 Macrophages - Extracted via **csRNA protocol** (Surface wash; captures only physically membrane-bound RNA).

## 2. Computational Alignment Pipeline
Raw sequencing reads were processed through a two-tier alignment strategy using **Bowtie2**:
1. **Primary Mapping (ncRNA):** Reads were first aligned against a custom RNAcentral database (inclusive of the 45S pre-ribosomal RNA sequence `NR_046233.2`) to capture non-coding regulatory elements.
2. **Secondary Mapping (Genome):** Unmapped reads from the primary alignment were subsequently routed and aligned to the standard Ensembl mouse genome (*mm39*).

## 3. Quantification
Raw BAM alignments were quantified into count matrices using `featureCounts`.

Initially, standard Differential Expression (via `edgeR`) was attempted to calculate the Log2 Fold Change between M1 and BMDM. However, the biological discrepancy in extraction methods (Trizol's massive internal rRNA/tRNA background vs. the sparse csRNA surface wash) skewed the denominators, resulting in universally negative `logFC` values. 

To correct for this, the mathematical approach was pivoted from **Relative Fold-Change** to **Absolute Relative Abundance (RPM - Reads Per Million)** strictly within the M1 surface sample. 

## 4. *In Silico* Ribo-Depletion & Target Discovery
An *in silico* ribo-depletion was performed using `pandas` to filter out the overwhelming `NR_046233.2` 45S rRNA spike (which accounted for ~87% of mapped reads). 

Analysis of the remaining secondary genomic mapping successfully identified highly abundant, physically surface-localized transcripts. These included:
* Validated transmembrane proteins (e.g., *Ccr5*, *Adam33*).
* Definitive markers of interferon-gamma M1 activation (e.g., *Ifit1*).
* Surface-anchored long non-coding RNAs (lncRNAs) (e.g., *Malat1*).

These high-RPM transcripts represent verified physical targets on the outer lipid bilayer of the M1 macrophage.

## Workflow Timeline

**1) Matrix Assembly** (`master_matrix.py`)
* **Action:** Merged the four raw `featureCounts` `.txt` files into a single, comprehensive dataset.
* **Output:** `master_counts.csv`

**2) Initial Statistical Testing** (`matrix_visual.r`)
* **Action:** Ingested the master matrix into `edgeR` to calculate Log2 Fold Change between the M1 and BMDM samples.
* **Output:** `M1_vs_BMDM_Differential_Expression.csv`

**3) Diagnostic Reality Check** (`upstream_filter.py`)
* **Action:** Filtered the `edgeR` results for upregulated non-coding RNAs.
* **Result:** Yielded an empty dataset (`M1_csRNA_Biomarkers.csv`). This mathematically proved the biological incompatibility of directly comparing a csRNA surface wash to a Trizol whole-cell extraction using Fold-Change.

**4) The Relative Abundance Pivot** (`rpm_calculation.py`)
* **Action:** Shifted the mathematical model from Relative Fold-Change to Absolute Relative Abundance (Reads Per Million). Calculated RPM strictly within the M1 sample to isolate true surface-localized transcripts.
* **Output:** `M1_Top_Surface_mRNAs.csv`

**5) Biological Translation** (`translate_genes.py`)
* **Action:** Queried the Ensembl REST API to convert the raw `ENSMUSG` identifiers into readable, biologically meaningful gene symbols and descriptions.
* **Output:** `M1_Translated_Surface_mRNAs.csv`

## Results & Visuals